In [12]:
# Equation for thermal conductivity from COMSOL for water 
def thermal_conductivity_water_comsol(T):
    return -0.869083936 + 0.00894880345*T - 1.58366345e-5*T**2 + 7.97543259e-9*T**3

print(thermal_conductivity_water_comsol(200)) 
   

0.35101483471999995


In [16]:
# properties of water 
k_water = 0.35 # thermal conductivity in W/(m*K)
rho_water = 1000 # density in kg/m^3
cp_water = 4200 # specific heat capacity in J/(kg*K)

# properties of carbon 
k_carbon = 1.12 # thermal conductivity in W/(m*K) from https://pubs.aip.org/aip/jap/article/88/11/6317/762375/Thermal-conductivity-of-amorphous-carbon-thin Figure 2 (film labelled L at 300K)
rho_carbon = 2300 # density in kg/m3 from https://pubs.aip.org/aip/jap/article/88/11/6317/762375/Thermal-conductivity-of-amorphous-carbon-thin Table 1 for film L with thickness 19nm
cp_carbon = 862 # specific heat carbon thin film from https://www.sciencedirect.com/science/article/pii/S0925963597002021 Figure 6, amorphous carbon 

# properties of gold 
k_gold = 50 # thermal conductivity from https://pmc.ncbi.nlm.nih.gov/articles/PMC9417296/#sec9 Figure 8 for Df=0.2*tf
rho_gold = 19.3 # using bulk density
cp_gold = 229 # J/(kg*K) from https://asmedigitalcollection.asme.org/heattransfer/article/137/5/051601/444642/Specific-Heat-Determination-of-Metallic-Thin-Films
# geometry 
t_carbon = 10e-9 # from Quantifoil website 
t_gold = 50e-9 # from https://www.science.org/doi/10.1126/science.1259530
t_water = 200e-9 

mesh_size = 100e-6


In [17]:
area_carbon = mesh_size * t_carbon
area_water = mesh_size * t_water
area_gold = mesh_size * t_gold

volume_carbon = mesh_size**2 * t_carbon
volume_water = mesh_size**2 * t_water
volume_gold = mesh_size**2 * t_gold

In [18]:
# calculate effective thermal conductivity using parallel resistance
k_effective = (k_carbon*t_carbon + k_water*t_water) / (t_carbon+t_water)
k_effective_gold = (k_gold*t_gold + k_water*t_water) / (t_gold+t_water)
# calculate effective density based on volume ratios 
rho_effective = (rho_carbon * volume_carbon + rho_water * volume_water) / (volume_water + volume_carbon)
rho_effective_gold = (rho_gold * volume_gold + rho_water * volume_water) / (volume_water + volume_gold)
# calculate effective specific heat based on mass ratios 
mass_carbon = rho_carbon * volume_carbon
mass_water = rho_water * volume_water
mass_gold = rho_gold * volume_gold

cp_effective = (cp_carbon * mass_carbon + cp_water * mass_water) / (mass_carbon + mass_water)
cp_effective_gold = (cp_gold * mass_gold + cp_water * mass_water) / (mass_gold + mass_water)


In [19]:
print("Effective thermal conductivity: {:.2f} W/(m*K)".format(k_effective))
print("Effective density: {:.2f} kg/m^3".format(rho_effective))
print("Effective specific heat capacity: {:.2f} J/(kg*K)".format(cp_effective))

print("Effective thermal conductivity with gold: {:.2f} W/(m*K)".format(k_effective_gold))
print("Effective density with gold: {:.2f} kg/m^3".format(rho_effective_gold))
print("Effective specific heat capacity with gold: {:.2f} J/(kg*K)".format(cp_effective_gold))

Effective thermal conductivity: 0.39 W/(m*K)
Effective density: 1061.90 kg/m^3
Effective specific heat capacity: 3855.72 J/(kg*K)
Effective thermal conductivity with gold: 10.28 W/(m*K)
Effective density with gold: 803.86 kg/m^3
Effective specific heat capacity with gold: 4180.93 J/(kg*K)


In [20]:
# compute diffusion time constant for water, carbon and effective material
def diffusion_time_constant(k, rho, cp, length):
    return (rho * cp * length**2) / k

diffusion_time_constant_water = diffusion_time_constant(k_water, rho_water, cp_water, mesh_size/2)
diffusion_time_constant_carbon = diffusion_time_constant(k_carbon, rho_carbon, cp_carbon, mesh_size/2)
diffusion_time_constant_effective = diffusion_time_constant(k_effective, rho_effective, cp_effective, mesh_size/2)
print("Diffusion time constant for water: {:.2f} ms".format(diffusion_time_constant_water*1000))
print("Diffusion time constant for carbon: {:.2f} ms".format(diffusion_time_constant_carbon*1000))
print("Diffusion time constant for effective material: {:.2f} ms".format(diffusion_time_constant_effective*1000))

Diffusion time constant for water: 30.00 ms
Diffusion time constant for carbon: 4.43 ms
Diffusion time constant for effective material: 26.47 ms
